# Analiza kardynalności cech w zbiorze danych Titanic

**Mateusz Michałek 426994**

**Cel ćwiczenia:** Zbadanie liczności unikalnych etykiet zmiennych jakościowych, podział cech ze względu na kardynalność oraz redukcja kardynalności zmiennej `cabin` wraz z oceną jej wpływu na dalsze modelowanie.

In [1]:
import numpy as np
import pandas as pd


def load_arff_to_df(filepath: str) -> pd.DataFrame:
    """Wczytuje plik .arff do obiektu pandas.DataFrame."""
    attributes = []
    data_start = 0

    with open(filepath, "r", encoding="utf-8") as file:
        for idx, line in enumerate(file):
            stripped = line.strip()
            lower_line = stripped.lower()
            if lower_line.startswith("@attribute"):
                col_name = stripped.split()[1].strip("'\"")
                attributes.append(col_name)
            elif lower_line.startswith("@data"):
                data_start = idx + 1
                break

    return pd.read_csv(
        filepath,
        skiprows=data_start,
        header=None,
        names=attributes,
        quotechar='"',
        skipinitialspace=True,
    )


FILE_PATH = "Titanic.arff"
df = load_arff_to_df(FILE_PATH).replace("?", np.nan)

## 1. Liczebność etykiet zmiennych jakościowych oraz liczba wszystkich pasażerów

In [5]:
qualitative_cols = [
    "pclass",
    "survived",
    "sex",
    "embarked",
    "boat",
    "cabin",
    "home.dest",
    "ticket",
    "name",
]

for col in qualitative_cols:
    print("Liczba etykiet zmiennej {}: {}".format(col, len(df[col].unique())))


print("\nLiczba wszystkich pasażerów: {}".format(len(df.index.unique())))

Liczba etykiet zmiennej pclass: 3
Liczba etykiet zmiennej survived: 2
Liczba etykiet zmiennej sex: 2
Liczba etykiet zmiennej embarked: 4
Liczba etykiet zmiennej boat: 28
Liczba etykiet zmiennej cabin: 187
Liczba etykiet zmiennej home.dest: 370
Liczba etykiet zmiennej ticket: 929
Liczba etykiet zmiennej name: 1307

Liczba wszystkich pasażerów: 1309


## 2. Podział zmiennych ze względu na kardynalność (moc zbioru)

W analizowanym zbiorze liczba wszystkich pasażerów wynosi 1309. Porównując liczbę unikalnych etykiet z całkowitą liczbą obserwacji, zmienne jakościowe można podzielić następująco:

* **Zmienne o małej kardynalności (niska moc zbioru):**
  * `survived` (2 etykiety), `sex` (2 etykiety), `pclass` (3 etykiety), `embarked` (4 etykiety z `NaN`).
  * Są łatwe do zakodowania (np. One-Hot Encoding) i interpretacji; każda kategoria ma dużą reprezentację w danych.
* **Zmienne o średniej kardynalności:**
  * `boat` (28 etykiet z `NaN`) – umiarkowana liczba numerów szalup ratunkowych.
* **Zmienne o dużej kardynalności (wysoka moc zbioru):**
  * `cabin` (187 etykiet), `home.dest` (370 etykiet), `ticket` (929 etykiet), `name` (1307 etykiet).
  * Kardynalność `name` i `ticket` jest zbliżona do liczby wszystkich pasażerów. Bezpośrednie użycie tych cech w modelach ML prowadziłoby do ogromnej wymiarowości, rzadkości macierzy i przeuczenia (overfitting).

## 3. Unikalne etykiety zmiennej `cabin`

In [3]:
cabin_unique = df["cabin"].unique()

print("Typ zwracanego obiektu:", type(cabin_unique))
print("Liczba unikalnych etykiet zmiennej cabin:", len(cabin_unique))
print("Przykładowe etykiety (pierwsze 20):", cabin_unique[:20])

Typ zwracanego obiektu: <class 'numpy.ndarray'>
Liczba unikalnych etykiet zmiennej cabin: 187
Przykładowe etykiety (pierwsze 20): ['B5' 'C22 C26' 'E12' 'D7' 'A36' 'C101' nan 'C62 C64' 'B35' 'A23'
 'B58 B60' 'D15' 'C6' 'D35' 'C148' 'C97' 'B49' 'C99' 'C52' 'T']


## 4. Redukcja kardynalności zmiennej cabin

In [4]:
df["CabinReduced"] = df["cabin"].astype(str).str[0]

# pierwsze 20 wierszy dla kolumn cabin i CabinReduced
display(df[["cabin", "CabinReduced"]].head(20))

# Zliczba etykiet przed i po redukcji oraz procent redukcji
n_cabin = len(df["cabin"].unique())
n_reduced = len(df["CabinReduced"].unique())
reduction_pct = (1 - n_reduced / n_cabin) * 100

print("Liczba etykiet zmiennej cabin: {}".format(n_cabin))
print("Liczba etykiet zmiennej CabinReduced: {}".format(n_reduced))
print("Redukcja kardynalności: {:.2f}%".format(reduction_pct))
print("Unikalne etykiety CabinReduced:", df["CabinReduced"].unique())

,cabin,CabinReduced
0,B5,B
1,C22 C26,C
2,C22 C26,C
3,C22 C26,C
4,C22 C26,C
5,E12,E
6,D7,D
7,A36,A
8,C101,C
9,NaN,n


Liczba etykiet zmiennej cabin: 187
Liczba etykiet zmiennej CabinReduced: 9
Redukcja kardynalności: 95.19%
Unikalne etykiety CabinReduced: ['B' 'C' 'E' 'D' 'A' 'n' 'T' 'F' 'G']


## 5. Uzasadnienie redukcji zmiennej `cabin`

**Uzasadnienie wyboru zmiennej `cabin` do redukcji**
*   Zmienna `cabin` ma wysoką kardynalność (~187 unikalnych wartości przy 1309 rekordach), ale w przeciwieństwie do `name` czy `ticket` posiada **czytelną strukturę domenową (`LL11`)**.
*   Pierwsza litera oznaczenia (`A`–`G`, `T`) oznacza pokład statku, który bezpośrednio wiąże się ze statusem ekonomicznym pasażera (klasą biletu) oraz fizyczną odległością od łodzi ratunkowych. Natomiast konkretne numery kabin (np. `22`, `58`, `101`) pojawiają się pojedynczo i wprowadzają szum.
*   Redukcja zmniejsza liczbę unikalnych etykiet o ponad 95% (ze 187 do 9 etykiet, gdzie `'n'` wynika z konwersji `NaN` przez `astype(str)` na `'nan'`).


**Korzyści w przyszłych analizach**
*  Znaczące ograniczenie wymiarowości po zakodowaniu zmiennych kategorycznych (np. One-Hot Encoding tworzy kilka kolumn zamiast blisko dwustu).
*   Większa liczba obserwacji przypadająca na każdą kategorię, co poprawia stabilność statystyczną i zdolność generalizacji modeli ML (mniejsze ryzyko przeuczenia).
*   Zachowanie informacji o pokładzie oraz o samym fakcie braku przypisanej kabiny (etykieta `'n'`).

**Potencjalne negatywne skutki:**
* Utrata szczegółowości informacji: w obrębie tego samego pokładu tracimy informację o dokładnym położeniu kabiny (dziób/rufa, burta) oraz o liczbie wykupionych kabin (np. zapisy wielokrotne typu `"C22 C26"` świadczące o zamożności lub licznej rodzinie).
* Artefakt konwersji `astype(str)`: wartość brakująca `NaN` zamieniana jest na napis `'nan'`, a po wycięciu pierwszego znaku staje się etykietą `'n'`. W tym zbiorze nie koliduje to z żadnym pokładem (`A`–`G`, `T`), jednak w innych zadaniach mogłoby dojść do przypadkowego połączenia braków z kategorią zaczynającą się na literę `'n'` (bezpieczniej w kodzie produkcyjnym najpierw użyć `fillna("Missing")`).